# Hybrid RAG Pipeline [Step 4 - Full Retrieval-Augmented Generation]

> **MLCourse - Agentic AI - Hybrid Search**

This notebook builds a complete RAG pipeline using hybrid retrieval.
We load Alice in Wonderland, chunk and index it with both BM25 and
dense vectors, retrieve relevant passages using RRF fusion, and
generate answers with ChatOllama.

In [1]:
# Import all libraries needed for this notebook.
import os                              # Environment and path handling
import re                              # Tokenization
import shutil                          # Directory cleanup

from rank_bm25 import BM25Okapi        # BM25 keyword search
import chromadb                        # Dense vector store
from chromadb.utils import embedding_functions  # Embedding models

from langchain_ollama import ChatOllama          # Local LLM
from langchain_core.prompts import ChatPromptTemplate  # Prompt templating
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output

In [2]:
# ## Part 1: Configuration
# Set up paths and model parameters.

CORPUS_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
CHROMA_DIR = r"D:\projects\python\MLCourse\03_agentic_ai\20_hybrid_search\chroma_rag"
RRF_K = 60           # RRF smoothing constant
TOP_K = 5            # Number of results to retrieve
CHUNK_SIZE = 500     # Target chunk size in characters
LLM_MODEL = "llama3.1:8b"
LLM_TEMP = 0         # Deterministic generation

print(f"LLM model: {LLM_MODEL}")
print(f"RRF k: {RRF_K}, Top K: {TOP_K}")

LLM model: llama3.1:8b
RRF k: 60, Top K: 5


In [3]:
# ## Part 2: Loading and Chunking the Corpus
# We load Alice in Wonderland and create overlapping chunks for indexing.

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Remove Gutenberg header/footer.
if "*** START OF" in raw_text:
    raw_text = raw_text.split("*** START OF", 1)[1]
if "*** END OF" in raw_text:
    raw_text = raw_text.split("*** END OF", 1)[0]

# Split into paragraphs.
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]

# Merge paragraphs into chunks of ~CHUNK_SIZE characters.
chunks = []
current = ""
for para in paragraphs:
    if len(current) + len(para) < CHUNK_SIZE:
        current += (" " if current else "") + para
    else:
        if current:
            chunks.append(current)
        current = para
if current:
    chunks.append(current)

print(f"Created {len(chunks)} chunks (target ~{CHUNK_SIZE} chars each)")
print(f"Avg chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")

Created 339 chunks (target ~500 chars each)
Avg chunk length: 423 chars


In [4]:
# ## Part 3: Building the BM25 Index
# BM25 provides fast keyword-based retrieval.

def simple_tokenize(text):
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_chunks = [simple_tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 index built: {len(chunks)} documents")

BM25 index built: 339 documents


In [5]:
# ## Part 4: Building the Dense Index
# ChromaDB provides semantic vector-based retrieval.

if os.path.exists(CHROMA_DIR):
    try:
        shutil.rmtree(CHROMA_DIR)
    except PermissionError:
        pass  # will recreate on top of existing

client = chromadb.PersistentClient(path=CHROMA_DIR)
ef = embedding_functions.DefaultEmbeddingFunction()
dense_collection = client.get_or_create_collection(
    name="alice_rag",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

BATCH_SIZE = 100
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    ids = [f"chunk_{j}" for j in range(i, i + len(batch))]
    dense_collection.add(documents=batch, ids=ids)

print(f"Dense index built: {dense_collection.count()} chunks")

Dense index built: 339 chunks


In [6]:
# ## Part 5: Hybrid Retrieval Function
# This function queries both BM25 and dense, then fuses with RRF.

def hybrid_retrieve(query, top_k=TOP_K, rrf_k=RRF_K):
    """Retrieve documents using hybrid BM25 + dense search with RRF.

    Args:
        query: the search query string.
        top_k: number of results to return.
        rrf_k: RRF smoothing constant.

    Returns:
        List of (document_text, rrf_score) tuples.
    """
    # BM25 retrieval.
    query_tokens = simple_tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_ranked = bm25_scores.argsort()[::-1]

    # Dense retrieval.
    dense_results = dense_collection.query(
        query_texts=[query], n_results=len(chunks)
    )
    dense_ids = dense_results["ids"][0]
    dense_ranked = [int(d.split("_")[1]) for d in dense_ids]

    # Reciprocal Rank Fusion.
    scores = {}
    for ranked_list in [bm25_ranked, dense_ranked]:
        for rank, doc_idx in enumerate(ranked_list, 1):
            if doc_idx not in scores:
                scores[doc_idx] = 0.0
            scores[doc_idx] += 1.0 / (rrf_k + rank)

    # Return top_k results.
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [(chunks[idx], score) for idx, score in fused]

# Test retrieval.
test_results = hybrid_retrieve("the white rabbit was late")
print(f"Test query: 'the white rabbit was late'")
for i, (doc, score) in enumerate(test_results[:3], 1):
    preview = doc[:60].replace("\n", " ")
    print(f"  #{i} (rrf={score:.6f}): {preview}...")

Test query: 'the white rabbit was late'
  #1 (rrf=0.032787): Alice was not a bit hurt, and she jumped up on to her feet i...
  #2 (rrf=0.032258): After a time she heard a little pattering of feet in the dis...
  #3 (rrf=0.029857): Alice watched the White Rabbit as he fumbled over the list, ...


In [7]:
# ## Part 6: RAG Prompt Template
# The prompt instructs the LLM to answer based on retrieved context only.

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the user's question using ONLY "
     "the provided context. If the context does not contain enough "
     "information to answer, say so. Be concise and cite specific details "
     "from the context when possible."),
    ("user",
     "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
])

In [8]:
# ## Part 7: Setting Up the LLM Chain
# We use ChatOllama with llama3.1:8b for local generation.

llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMP)
chain = RAG_PROMPT | llm | StrOutputParser()

print(f"LLM chain ready: {LLM_MODEL} (temp={LLM_TEMP})")

LLM chain ready: llama3.1:8b (temp=0)


In [9]:
# ## Part 8: The Full RAG Pipeline
# Combine retrieval and generation into a single function.

def rag_query(question, top_k=TOP_K):
    """Complete RAG pipeline: retrieve + generate.

    Args:
        question: user's natural language question.
        top_k: number of context passages to retrieve.

    Returns:
        Dictionary with the answer, sources, and scores.
    """
    # Retrieve relevant passages.
    results = hybrid_retrieve(question, top_k=top_k)

    # Build context string from retrieved passages.
    context_parts = []
    for i, (doc, score) in enumerate(results, 1):
        context_parts.append(f"[Passage {i}]\n{doc}")
    context = "\n\n".join(context_parts)

    # Generate answer with LLM.
    answer = chain.invoke({
        "context": context,
        "question": question,
    })

    return {
        "answer": answer,
        "sources": [(doc[:100].replace("\n", " "), score) for doc, score in results],
        "context_count": len(results),
    }

In [10]:
# ## Part 9: Running RAG Queries
# Let us test the pipeline with several questions about Alice in Wonderland.

questions = [
    "What did the White Rabbit say when he was late?",
    "What happened at the Queen's croquet ground?",
    "Who is the Cheshire Cat and what did he say?",
]

for q in questions:
    print(f"{'='*70}")
    print(f"Q: {q}")
    print()
    result = rag_query(q)
    print(f"A: {result['answer']}")
    print()
    print("Sources used:")
    for i, (preview, score) in enumerate(result["sources"][:3], 1):
        print(f"  #{i} (rrf={score:.6f}): {preview}...")
    print()

Q: What did the White Rabbit say when he was late?



A: When the White Rabbit was late, he said "Oh my ears and whiskers, how late it's getting!" (Passage 3)

Sources used:
  #1 (rrf=0.032522): “It’s—it’s a very fine day!” said a timid voice at her side. She was walking by the White Rabbit, wh...
  #2 (rrf=0.032018): After a time she heard a little pattering of feet in the distance, and she hastily dried her eyes to...
  #3 (rrf=0.031281): Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was al...

Q: What happened at the Queen's croquet ground?



A: The game began after people got settled down in their places. Alice thought she had never seen such a curious croquet-ground in her life; it was all ridges and furrows; the balls were live hedgehogs, the mallets live flamingoes, and the soldiers had to double themselves up and stand on their hands and feet to make the arches (Passage 4).

Sources used:
  #1 (rrf=0.032002): CHAPTER VIII. The Queen’s Croquet-Ground A large rose-tree stood near the entrance of the garden: th...
  #2 (rrf=0.031778): “A fine day, your Majesty!” the Duchess began in a low, weak voice. “Now, I give you fair warning,” ...
  #3 (rrf=0.031778): “Their heads are gone, if it please your Majesty!” the soldiers shouted in reply. “That’s right!” sh...

Q: Who is the Cheshire Cat and what did he say?



A: The Cheshire Cat is a cat that can grin, as mentioned in Passage 1. However, it does not speak until later passages. In Passage 3, the Cheshire Cat speaks to Alice in a low voice, but its exact words are not specified.

Sources used:
  #1 (rrf=0.032266): “It’s a Cheshire cat,” said the Duchess, “and that’s why. Pig!” She said the last word with such sud...
  #2 (rrf=0.031545): When she got back to the Cheshire Cat, she was surprised to find quite a large crowd collected round...
  #3 (rrf=0.031514): “How do you like the Queen?” said the Cat in a low voice. “Not at all,” said Alice: “she’s so extrem...



In [11]:
# ## Part 10: Comparing Hybrid vs Single-Mode Retrieval
# Let us see what happens when we use only BM25 or only dense retrieval.

def bm25_only_retrieve(query, top_k=TOP_K):
    """Retrieve using BM25 only (no dense, no fusion)."""
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked = scores.argsort()[::-1][:top_k]
    return [(chunks[idx], scores[idx]) for idx in ranked]

def dense_only_retrieve(query, top_k=TOP_K):
    """Retrieve using dense vectors only (no BM25, no fusion)."""
    results = dense_collection.query(query_texts=[query], n_results=top_k)
    ids = results["ids"][0]
    return [(chunks[int(d.split("_")[1])], 1 - results["distances"][0][i])
            for i, d in enumerate(ids)]

test_q = "a creature that smiles"
print(f"Query: '{test_q}'")
print()

print("BM25 only (keyword matching):")
bm25_res = bm25_only_retrieve(test_q, 2)
for doc, score in bm25_res:
    print(f"  {doc[:70].replace(chr(10), ' ')}...")

print()
print("Dense only (semantic matching):")
dense_res = dense_only_retrieve(test_q, 2)
for doc, score in dense_res:
    print(f"  {doc[:70].replace(chr(10), ' ')}...")

print()
print("Hybrid (RRF fusion):")
hybrid_res = hybrid_retrieve(test_q, 2)
for doc, score in hybrid_res:
    print(f"  {doc[:70].replace(chr(10), ' ')}...")

Query: 'a creature that smiles'

BM25 only (keyword matching):
  Alice caught the baby with some difficulty, as it was a queer-shaped l...
  Alice was just beginning to think to herself, “Now, what am I to do wi...

Dense only (semantic matching):
  “How doth the little crocodile     Improve his shining tail, And pour ...
  “Did you say pig, or fig?” said the Cat. “I said pig,” replied Alice; ...

Hybrid (RRF fusion):


  Alice caught the baby with some difficulty, as it was a queer-shaped l...
  Alice was just beginning to think to herself, “Now, what am I to do wi...


In [12]:
# ## Part 11: Pipeline Summary
# Our hybrid RAG pipeline combines the best of both worlds:
#
# 1. BM25 handles exact keyword matches and rare terms.
# 2. Dense retrieval captures semantic meaning and synonyms.
# 3. RRF merges results without needing to tune weights.
# 4. The LLM generates a coherent answer from retrieved context.
#
# This architecture is production-ready and can be extended with:
# - Re-ranking models for further result refinement.
# - Metadata filtering for structured queries.
# - Conversation memory for multi-turn interactions.
# - Multiple embedding models for domain-specific retrieval.

print("Hybrid RAG Pipeline Summary:")
print("  1. Load and chunk documents")
print("  2. Index with BM25 (keyword) and ChromaDB (dense)")
print("  3. Query both systems in parallel")
print("  4. Merge results with Reciprocal Rank Fusion")
print("  5. Pass top-k context to LLM for answer generation")
print()
print("Advantages over single-mode retrieval:")
print("  + Handles both keyword and semantic queries")
print("  + More robust to vocabulary mismatches")
print("  + No weight tuning needed (RRF is parameter-free)")
print("  + Works with any combination of retrieval systems")

Hybrid RAG Pipeline Summary:
  1. Load and chunk documents
  2. Index with BM25 (keyword) and ChromaDB (dense)
  3. Query both systems in parallel
  4. Merge results with Reciprocal Rank Fusion
  5. Pass top-k context to LLM for answer generation

Advantages over single-mode retrieval:
  + Handles both keyword and semantic queries
  + More robust to vocabulary mismatches
  + No weight tuning needed (RRF is parameter-free)
  + Works with any combination of retrieval systems


In [13]:
# ## Part 12: Cleanup
# Remove the temporary ChromaDB directory.

if os.path.exists(CHROMA_DIR):
    try:
        shutil.rmtree(CHROMA_DIR)
        print("Cleaned up ChromaDB directory")
    except PermissionError:
        print("ChromaDB directory locked by another process; skipping cleanup")

print("Notebook complete.")

ChromaDB directory locked by another process; skipping cleanup
Notebook complete.
